# Deep Learning (Fall 2026) — Project 1: Custom Sub-5M Parameter ResNet on CIFAR-10

This notebook implements our **100% from-scratch Residual Network (`CustomResNet`)** and training pipeline for CIFAR-10 under the strict **5,000,000 trainable parameter ceiling**, integrated directly with the competition image folders (`train_images/`, `test_images/`) and CSV metadata files (`train_labels.csv`, `test.csv`, `sample_submission.csv`).

### Key Highlights
- **Built from First Principles:** Directly implements $\text{Output} = \text{ReLU}(S(x) + F(x))$ where $F(x)$ is `Conv2d -> BatchNorm2d -> ReLU -> Conv2d -> BatchNorm2d` and $S(x)$ is the identity or $K_i \times K_i$ projection shortcut.
- **Explicit Design Variables ($N, B_i, C_i, F_i, K_i, P$):** Exposes stage count $N$, blocks per stage $B_i$, channel widths $C_i$, conv filter sizes $F_i$, skip kernel sizes $K_i$, and average pooling size $P$.
- **Champion Sub-5M Architecture (`champion_sub5m`):** Uses $N=4$ stages, $C = [64, 128, 208, 264]$, $B = [3, 3, 2, 2]$, $F = [3, 3, 3, 3]$, $K = [1, 1, 1, 1]$, and $P = 4$ for **4,917,450 trainable parameters** ($98.35\%$ of the 5M budget).
- **Native Folder + CSV Integration:** Loads training images from `train_images/` + `train_labels.csv` (`id,filename,label`) with a stratified train/validation split (or falls back automatically to `torchvision.datasets.CIFAR10`), and evaluates `test_images/` + `test.csv` (`id,filename`) to export a 2,000-row `submission.csv` (`id,label`).

In [ ]:
!pip install -q torchsummary pandas matplotlib Pillow

In [ ]:
import csv, json, math, os, pickle, random, time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional, Callable, Union

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
from torchsummary import summary

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

def seed_everything(seed: int = 2026) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

seed_everything(2026)

## 1. From-Scratch Residual Network Architecture ($N, B_i, C_i, F_i, K_i, P$)
Each `ResidualUnit` computes $\text{ReLU}(S(x) + F(x))$, and `CustomResNet` chains $N$ stages of $B_i$ residual blocks followed by $P \times P$ average pooling and a linear classification head.

In [ ]:
@dataclass
class ResNetSpec:
    name: str
    stem_channels: int
    channels: List[int]          # C_i: channels in Residual Layer i
    blocks_per_stage: List[int]  # B_i: residual blocks in Residual Layer i
    conv_kernels: List[int]      # F_i: conv kernel size in Residual Layer i
    skip_kernels: List[int]      # K_i: skip connection kernel size in Residual Layer i
    pool_size: int               # P:   average pool kernel size
    dropout_rate: float = 0.1
    num_classes: int = 10

    def to_dict(self) -> dict:
        return asdict(self)

class ResidualUnit(nn.Module):
    """Implements ReLU(S(x) + F(x)) from scratch."""
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, conv_kernel: int = 3, skip_kernel: int = 1):
        super().__init__()
        conv_pad = conv_kernel // 2
        self.residual_branch = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=conv_kernel, stride=stride, padding=conv_pad, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=conv_kernel, stride=1, padding=conv_pad, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        if stride != 1 or in_channels != out_channels:
            skip_pad = skip_kernel // 2
            self.skip_connection = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=skip_kernel, stride=stride, padding=skip_pad, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.skip_connection = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.relu(self.skip_connection(x) + self.residual_branch(x), inplace=True)

class CustomResNet(nn.Module):
    """Configurable N-Stage Residual Network built from scratch for 32x32 CIFAR-10 images."""
    def __init__(self, spec: ResNetSpec):
        super().__init__()
        self.spec = spec
        self.current_channels = spec.stem_channels

        self.stem = nn.Sequential(
            nn.Conv2d(3, spec.stem_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(spec.stem_channels),
            nn.ReLU(inplace=True),
        )

        stages = []
        for idx in range(len(spec.channels)):
            stage_stride = 1 if idx == 0 else 2
            stages.append(self._build_stage(
                out_channels=spec.channels[idx],
                num_blocks=spec.blocks_per_stage[idx],
                first_stride=stage_stride,
                conv_kernel=spec.conv_kernels[idx],
                skip_kernel=spec.skip_kernels[idx],
            ))
        self.residual_stages = nn.Sequential(*stages)
        self.pool_size = spec.pool_size
        self.dropout = nn.Dropout(p=spec.dropout_rate) if spec.dropout_rate > 0 else nn.Identity()

        final_map_width = 32 // (2 ** (len(spec.channels) - 1))
        pooled_width = max(1, final_map_width // spec.pool_size)
        fc_input_dim = spec.channels[-1] * (pooled_width ** 2)
        self.classifier = nn.Linear(fc_input_dim, spec.num_classes)
        self._init_weights()

    def _build_stage(self, out_channels, num_blocks, first_stride, conv_kernel, skip_kernel):
        blocks = []
        for b_idx in range(num_blocks):
            stride = first_stride if b_idx == 0 else 1
            blocks.append(ResidualUnit(self.current_channels, out_channels, stride, conv_kernel, skip_kernel))
            self.current_channels = out_channels
        return nn.Sequential(*blocks)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.01); nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.residual_stages(x)
        x = F.avg_pool2d(x, kernel_size=self.pool_size)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(self.dropout(x))

def count_trainable_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 2. Parameter Count Verification (`<= 5,000,000`) & `torchsummary`
We compare four custom sub-5M ResNet designs and verify our **Champion Sub-5M Model (`4,917,450` parameters)** via `torchsummary.summary`.

In [ ]:
ARCHITECTURE_REGISTRY = {
    "baseline_4stage": ResNetSpec(
        name="baseline_4stage", stem_channels=64, channels=[64, 128, 192, 256],
        blocks_per_stage=[2, 2, 2, 2], conv_kernels=[3, 3, 3, 3], skip_kernels=[1, 1, 1, 1],
        pool_size=4, dropout_rate=0.0
    ),
    "wide_3stage": ResNetSpec(
        name="wide_3stage", stem_channels=64, channels=[64, 144, 264],
        blocks_per_stage=[3, 3, 3], conv_kernels=[3, 3, 3], skip_kernels=[1, 1, 1],
        pool_size=8, dropout_rate=0.05
    ),
    "skip3x3_4stage": ResNetSpec(
        name="skip3x3_4stage", stem_channels=64, channels=[64, 128, 192, 256],
        blocks_per_stage=[2, 2, 2, 2], conv_kernels=[3, 3, 3, 3], skip_kernels=[1, 3, 3, 3],
        pool_size=4, dropout_rate=0.05
    ),
    "champion_sub5m": ResNetSpec(
        name="champion_sub5m", stem_channels=64, channels=[64, 128, 208, 264],
        blocks_per_stage=[3, 3, 2, 2], conv_kernels=[3, 3, 3, 3], skip_kernels=[1, 1, 1, 1],
        pool_size=4, dropout_rate=0.1
    ),
}

rows = []
for name, spec in ARCHITECTURE_REGISTRY.items():
    net = CustomResNet(spec)
    params = count_trainable_parameters(net)
    assert params <= 5_000_000, f"{name} exceeds 5M parameters!"
    rows.append({
        "Architecture": name,
        "N (Stages)": len(spec.channels),
        "C_i (Channels)": str(spec.channels),
        "B_i (Blocks)": str(spec.blocks_per_stage),
        "F_i (Filters)": str(spec.conv_kernels),
        "K_i (Skip)": str(spec.skip_kernels),
        "P (Pool)": spec.pool_size,
        "Trainable Parameters": f"{params:,}",
        "% of 5M Budget": f"{100.0 * params / 5_000_000:.2f}%"
    })
display(pd.DataFrame(rows))

champion_net = CustomResNet(ARCHITECTURE_REGISTRY["champion_sub5m"]).to(DEVICE)
print(f"\nChampion Model Trainable Parameters: {count_trainable_parameters(champion_net):,} (Strictly <= 5,000,000)")
summary(champion_net, input_size=(3, 32, 32), device=str(DEVICE))

## 3. Dataset Pipeline: `train_images/` + `train_labels.csv` (with Automatic Fallback)
- **`CIFARCSVDataset`**: Connects `train_labels.csv` (`id,filename,label`, 50,000 rows) with `train_images/train_00000.png .. train_49999.png` and `test.csv` (`id,filename`, 2,000 rows) with `test_images/test_0000.png .. test_1999.png`.
- Performs a **stratified 90/10 train/validation split** when loading from `train_images/` + `train_labels.csv`, or automatically falls back to `torchvision.datasets.CIFAR10` if `train_images/` is not present.

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)
CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
BATCH_SIZE   = 128

train_transform = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode='reflect'),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    T.RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3), value=0.0),
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

class CIFARCSVDataset(Dataset):
    """Links `train_labels.csv` (`id,filename,label`) or `test.csv` (`id,filename`) with an image folder."""
    def __init__(self, data_source: Union[str, Path, pd.DataFrame], img_dir: Union[str, Path], transform: Optional[Callable] = None, is_test: bool = False):
        self.df = data_source.reset_index(drop=True).copy() if isinstance(data_source, pd.DataFrame) else pd.read_csv(data_source)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.is_test = is_test
        self.classes = CIFAR10_CLASSES

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        fname = str(row['filename']).strip() if 'filename' in row else (f"test_{int(row['id']):04d}.png" if self.is_test else f"train_{int(row['id']):05d}.png")
        image = Image.open(self.img_dir / fname).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        if self.is_test:
            return int(row['id']), image
        return image, int(row['label'])

def stratified_train_val_split(df: pd.DataFrame, val_split: float = 0.1, seed: int = 2026) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed)
    train_idx, val_idx = [], []
    for _, grp in df.groupby('label'):
        idxs = grp.index.to_numpy().copy()
        rng.shuffle(idxs)
        n_val = max(1, int(round(len(idxs) * val_split)))
        val_idx.extend(idxs[:n_val].tolist())
        train_idx.extend(idxs[n_val:].tolist())
    return df.loc[train_idx].sample(frac=1.0, random_state=seed).reset_index(drop=True), df.loc[val_idx].sort_values('id').reset_index(drop=True)

TRAIN_CSV_PATH = Path('./train_labels.csv')
TRAIN_IMG_DIR  = Path('./train_images')

if TRAIN_CSV_PATH.exists() and TRAIN_IMG_DIR.exists():
    full_df = pd.read_csv(TRAIN_CSV_PATH)
    train_df, val_df = stratified_train_val_split(full_df, val_split=0.1, seed=2026)
    train_set = CIFARCSVDataset(train_df, TRAIN_IMG_DIR, transform=train_transform, is_test=False)
    test_set  = CIFARCSVDataset(val_df, TRAIN_IMG_DIR, transform=eval_transform, is_test=False)
    print(f"Loaded custom CSV + folder dataset: {len(train_set):,} train | {len(test_set):,} validation images")
else:
    train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
    test_set  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=eval_transform)
    print(f"Loaded torchvision CIFAR-10 fallback: {len(train_set):,} train | {len(test_set):,} test images")

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2 if DEVICE.type == 'cuda' else 0, pin_memory=(DEVICE.type == 'cuda'))
test_loader  = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2 if DEVICE.type == 'cuda' else 0, pin_memory=(DEVICE.type == 'cuda'))

## 4. From-Scratch Training & Evaluation Engine

In [ ]:
@torch.no_grad()
def evaluate_network(model: nn.Module, loader: DataLoader, loss_fn: nn.Module, apply_tta: bool = False) -> Tuple[float, float]:
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = model(images)
        if apply_tta:
            logits = 0.5 * (logits + model(torch.flip(images, dims=[3])))
        loss = loss_fn(logits, labels)
        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += labels.size(0)
    return total_loss / total_samples, 100.0 * total_correct / total_samples

def train_and_evaluate(spec: ResNetSpec, epochs: int = 80, peak_lr: float = 0.1, weight_decay: float = 5e-4, label_smoothing: float = 0.05, out_dir: Path = Path('./outputs')):
    seed_everything(2026)
    out_dir.mkdir(parents=True, exist_ok=True)
    model = CustomResNet(spec).to(DEVICE)
    param_count = count_trainable_parameters(model)
    assert param_count <= 5_000_000, f"Model exceeds 5M parameters: {param_count:,}"

    train_loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    eval_loss_fn  = nn.CrossEntropyLoss()

    matrix_params = [p for p in model.parameters() if p.requires_grad and p.ndim >= 2]
    vector_params = [p for p in model.parameters() if p.requires_grad and p.ndim < 2]
    optimizer = optim.SGD([
        {'params': matrix_params, 'weight_decay': weight_decay},
        {'params': vector_params, 'weight_decay': 0.0},
    ], lr=peak_lr, momentum=0.9, nesterov=True)

    warmup = min(5, max(1, epochs // 10))
    scheduler = optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda e: float(e + 1) / warmup if e < warmup else 0.5 * (1.0 + math.cos(math.pi * (e - warmup) / max(1, epochs - warmup)))
    )
    use_amp = (DEVICE.type == 'cuda')
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    history = []
    best_acc = 0.0
    best_ckpt = out_dir / f"{spec.name}_best.pth"
    t0 = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        run_loss, run_correct, run_total = 0.0, 0, 0
        cur_lr = optimizer.param_groups[0]['lr']
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            if use_amp:
                with torch.autocast('cuda', dtype=torch.float16):
                    logits = model(images)
                    loss = train_loss_fn(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(images)
                loss = train_loss_fn(logits, labels)
                loss.backward()
                optimizer.step()
            run_loss += loss.item() * labels.size(0)
            run_correct += (logits.argmax(dim=1) == labels).sum().item()
            run_total += labels.size(0)

        scheduler.step()
        tr_loss, tr_acc = run_loss / run_total, 100.0 * run_correct / run_total
        te_loss, te_acc = evaluate_network(model, test_loader, eval_loss_fn, apply_tta=False)
        if te_acc > best_acc:
            best_acc = te_acc
            torch.save({'model_state_dict': model.state_dict(), 'spec': spec.to_dict(), 'best_test_acc': best_acc, 'trainable_parameters': param_count}, best_ckpt)
        history.append({'epoch': epoch, 'lr': cur_lr, 'train_loss': tr_loss, 'train_acc': tr_acc, 'test_loss': te_loss, 'test_acc': te_acc, 'best_test_acc': best_acc})
        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(f"[{spec.name}] Epoch {epoch:03d}/{epochs:03d} (LR={cur_lr:.4f}) | Train Loss: {tr_loss:.4f} ({tr_acc:.2f}%) | Test Loss: {te_loss:.4f} | Test Acc: {te_acc:.2f}% (Best: {best_acc:.2f}%)")

    elapsed = time.perf_counter() - t0
    ckpt = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    _, std_acc = evaluate_network(model, test_loader, eval_loss_fn, apply_tta=False)
    _, tta_acc = evaluate_network(model, test_loader, eval_loss_fn, apply_tta=True)

    print("\n" + "=" * 76)
    print(f"FINAL RESULTS — {spec.name}")
    print(f"  Total Trainable Parameters : {param_count:,} (<= 5,000,000 limit)")
    print(f"  Final Test Accuracy (Std)  : {std_acc:.2f}%")
    print(f"  Final Test Accuracy (TTA)  : {tta_acc:.2f}%")
    print(f"  Total Training Time        : {elapsed:.1f}s ({elapsed/60.0:.2f} min)")
    print("=" * 76)
    return {'model': model, 'spec': spec.to_dict(), 'parameters': param_count, 'best_test_acc': std_acc, 'best_test_acc_tta': tta_acc, 'training_seconds': elapsed, 'history': history}

## 5. Train Champion Sub-5M ResNet & Generate Report Plots

In [ ]:
EPOCHS = 80  # Set to 100-150 for maximum accuracy (~95.5%)
OUT_DIR = Path('./outputs')

champ_spec = ARCHITECTURE_REGISTRY['champion_sub5m']
champ_res = train_and_evaluate(champ_spec, epochs=EPOCHS, peak_lr=0.1, weight_decay=5e-4, label_smoothing=0.05, out_dir=OUT_DIR)

In [ ]:
history = champ_res['history']
epochs_list = [h['epoch'] for h in history]
best_acc = champ_res['best_test_acc']
params = champ_res['parameters']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].plot(epochs_list, [h['train_loss'] for h in history], label='Train Loss', color='#1f77b4', lw=2)
axes[0].plot(epochs_list, [h['test_loss'] for h in history], label='Test Loss', color='#d62728', lw=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title(f'Champion Sub-5M ResNet Loss Curves ({params:,} params)')
axes[0].grid(True, linestyle='--', alpha=0.4); axes[0].legend()

axes[1].plot(epochs_list, [h['train_acc'] for h in history], label='Train Accuracy (%)', color='#1f77b4', lw=2)
axes[1].plot(epochs_list, [h['test_acc'] for h in history], label=f'Test Accuracy (Best: {best_acc:.2f}%)', color='#2ca02c', lw=2)
axes[1].axhline(90.0, color='gray', linestyle=':', label='90% Target Threshold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Champion Sub-5M ResNet Accuracy Progression')
axes[1].grid(True, linestyle='--', alpha=0.4); axes[1].legend()

fig.tight_layout()
fig.savefig(OUT_DIR / 'champion_sub5m_training_curves.png', dpi=160)
plt.show()

# Per-Class Accuracy Bar Chart
model_eval = champ_res['model'].eval()
hits, counts = [0]*10, [0]*10
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = 0.5 * (model_eval(images) + model_eval(torch.flip(images, dims=[3])))
        for y_t, y_p in zip(labels, logits.argmax(dim=1)):
            c = int(y_t.item()); counts[c] += 1
            if int(y_p.item()) == c: hits[c] += 1

classes = getattr(test_set, 'classes', CIFAR10_CLASSES)
accs = [100.0 * hits[i] / max(1, counts[i]) for i in range(10)]
fig, ax = plt.subplots(figsize=(9.5, 4.2))
bars = ax.bar(classes, accs, color='#2b5c8f')
ax.axhline(champ_res['best_test_acc_tta'], color='#d95f02', linestyle='--', label=f"Overall TTA Acc ({champ_res['best_test_acc_tta']:.2f}%)")
ax.set_ylim(max(0.0, min(accs) - 10.0), 100.0)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Per-Class CIFAR-10 Test Accuracy (Champion Sub-5M ResNet)')
ax.grid(axis='y', linestyle='--', alpha=0.4); ax.legend()
for b in bars:
    h = b.get_height()
    ax.annotate(f'{h:.1f}%', xy=(b.get_x() + b.get_width()/2, h), xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=8.5)
fig.tight_layout()
fig.savefig(OUT_DIR / 'champion_sub5m_per_class_accuracy.png', dpi=160)
plt.show()

print(f"FINAL TEST ACCURACY: {champ_res['best_test_acc']:.2f}% (Standard) | {champ_res['best_test_acc_tta']:.2f}% (TTA)")
print(f"NUMBER OF PARAMETERS: {champ_res['parameters']:,}")

## 6. Generate Kaggle Competition `submission.csv` (`id,label`) from `test_images/` + `test.csv`
Reads `test.csv` (`id,filename`, 2,000 rows: `test_0000.png` through `test_1999.png`) and loads each PNG from `test_images/` to generate `submission.csv` (`id,label`) matching `sample_submission.csv`.

In [ ]:
TEST_CSV_PATH   = Path('./test.csv')               # 2,000 rows: id,filename
TEST_IMG_DIR    = Path('./test_images')            # Folder with test_0000.png .. test_1999.png
SAMPLE_SUB_FILE = Path('./sample_submission.csv')  # Template with id,label

model_eval = champ_res['model'].eval()

if TEST_CSV_PATH.exists() and TEST_IMG_DIR.exists():
    kaggle_test_set = CIFARCSVDataset(TEST_CSV_PATH, TEST_IMG_DIR, transform=eval_transform, is_test=True)
    kaggle_test_loader = DataLoader(
        kaggle_test_set, batch_size=256, shuffle=False,
        num_workers=2 if DEVICE.type == 'cuda' else 0, pin_memory=(DEVICE.type == 'cuda')
    )
    ids, preds = [], []
    with torch.no_grad():
        for batch_ids, images in kaggle_test_loader:
            images = images.to(DEVICE)
            # Horizontal-flip Test-Time Augmentation (TTA)
            logits = 0.5 * (model_eval(images) + model_eval(torch.flip(images, dims=[3])))
            ids.extend([int(i) for i in batch_ids])
            preds.extend(logits.argmax(dim=1).cpu().tolist())

    sub_df = pd.DataFrame({'id': np.asarray(ids, dtype=int), 'label': np.asarray(preds, dtype=int)})
    assert list(sub_df.columns) == ['id', 'label']
    sub_df.to_csv('submission.csv', index=False)
    sub_df.to_csv(OUT_DIR / 'submission.csv', index=False)
    print(f'Saved submission.csv from test_images/ + test.csv ({len(sub_df)} rows, columns: id,label)')
    print(sub_df.head(10).to_string(index=False))
else:
    # Fallback preview: write first 2,000 test predictions matching sample_submission.csv IDs 0..1999
    preds = []
    with torch.no_grad():
        for images, _ in test_loader:
            b = images.to(DEVICE)
            logits = 0.5 * (model_eval(b) + model_eval(torch.flip(b, dims=[3])))
            preds.extend(logits.argmax(dim=1).cpu().tolist())
    sub_df = pd.DataFrame({'id': list(range(2000)), 'label': preds[:2000]})
    sub_df.to_csv('submission.csv', index=False)
    sub_df.to_csv(OUT_DIR / 'submission.csv', index=False)
    print('Saved fallback submission.csv (2,000 rows, columns: id,label)')
    print(sub_df.head(10).to_string(index=False))